# 04 — Scaling Analysis

How does performance scale with dataset size and number of features?
When do Foundation Models start outperforming GBDTs?

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.registry import load_dataset, get_all_dataset_names

sns.set_theme(style='whitegrid')
plt.rcParams.update({'font.size': 12, 'axes.labelsize': 14, 'axes.titlesize': 14, 'savefig.dpi': 300})

os.makedirs('../results/figures', exist_ok=True)

%matplotlib inline

test_df = pd.read_csv('../results/aggregated/test_results.csv')

In [ ]:
# Collect dataset sizes
dataset_sizes = {}
for name in get_all_dataset_names():
    try:
        X, y, info = load_dataset(name)
        dataset_sizes[name] = {'n_samples': len(X), 'n_features': X.shape[1]}
    except Exception:
        pass

sizes_df = pd.DataFrame(dataset_sizes).T
sizes_df.index.name = 'dataset'
sizes_df = sizes_df.reset_index()

# Merge with results
merged = test_df.merge(sizes_df, on='dataset')

In [ ]:
# Performance vs dataset size — grouped by model family
model_families = {
    'xgboost': 'GBDT', 'lightgbm': 'GBDT', 'catboost': 'GBDT',
    'ft_transformer': 'DL', 'tabnet': 'DL', 'saint': 'DL',
    'tabm': 'DL', 'mlp': 'DL', 'realmlp': 'DL',
    'tabpfn': 'FM'
}
merged['family'] = merged['model'].map(model_families)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Classification
clf = merged[merged['task_type'].isin(['binary', 'multiclass'])]
if not clf.empty and 'accuracy' in clf.columns:
    for family, group in clf.groupby('family'):
        axes[0].scatter(group['n_samples'], group['accuracy'],
                       label=family, s=60, alpha=0.7)
    axes[0].set_xscale('log')
    axes[0].set_xlabel('Number of Samples (log scale)')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Classification: Accuracy vs Dataset Size')
    axes[0].legend()

# Regression
reg = merged[merged['task_type'] == 'regression']
if not reg.empty and 'r2' in reg.columns:
    for family, group in reg.groupby('family'):
        axes[1].scatter(group['n_samples'], group['r2'],
                       label=family, s=60, alpha=0.7)
    axes[1].set_xscale('log')
    axes[1].set_xlabel('Number of Samples (log scale)')
    axes[1].set_ylabel('R²')
    axes[1].set_title('Regression: R² vs Dataset Size')
    axes[1].legend()

plt.tight_layout()
# Save as PNG and PDF
plt.savefig('../results/figures/scaling_samples.png', dpi=300, bbox_inches='tight')
plt.savefig('../results/figures/scaling_samples.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Performance vs number of features
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (task_label, subset, metric) in zip(axes, [
    ('Classification', merged[merged['task_type'].isin(['binary', 'multiclass'])], 'accuracy'),
    ('Regression', merged[merged['task_type'] == 'regression'], 'r2'),
]):
    if subset.empty or metric not in subset.columns:
        continue
    for family, group in subset.groupby('family'):
        ax.scatter(group['n_features'], group[metric],
                  label=family, s=60, alpha=0.7)
    ax.set_xlabel('Number of Features')
    ax.set_ylabel(metric.upper())
    ax.set_title(f'{task_label}: {metric} vs Feature Count')
    ax.legend()

plt.tight_layout()
# Save as PNG and PDF
plt.savefig('../results/figures/scaling_features.png', dpi=300, bbox_inches='tight')
plt.savefig('../results/figures/scaling_features.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Performance gap: best GBDT vs best DL/Foundation per dataset
for task_type, metric, higher_is_better in [
    ('binary', 'roc_auc', True),
    ('multiclass', 'accuracy', True),
    ('regression', 'rmse', False),
]:
    subset = merged[merged['task_type'] == task_type]
    if subset.empty or metric not in subset.columns:
        continue

    # Best per family per dataset
    agg_fn = 'max' if higher_is_better else 'min'
    best = subset.groupby(['dataset', 'family'])[metric].agg(agg_fn).unstack('family')

    print(f"\n=== {task_type}: Best {metric} per family ===")
    display(best)

    if 'GBDT' in best.columns and 'DL' in best.columns:
        gap = best['DL'] - best['GBDT'] if higher_is_better else best['GBDT'] - best['DL']
        print(f"\nDL advantage (positive = DL wins):")
        print(gap.sort_values(ascending=False))

## True Learning Curves

How does performance change as we increase the fraction of training data available?
This reveals whether models are data-hungry (steep improvement with more data) or data-efficient (plateau early).

**Expected data format:** `results/raw/<dataset>/<model>/learning_curve.json`
Each file should contain a dict with keys `fractions` and `scores`, e.g.:
```json
{"fractions": [0.1, 0.2, 0.4, 0.6, 0.8, 1.0], "scores": [0.72, 0.76, 0.80, 0.82, 0.84, 0.85]}
```

If this file does not exist yet, the code below prints a skip message.
To generate it: modify `scripts/train.py` to train at each fraction and save the dict.

In [ ]:
import json
import glob as glob_module

# Training fractions to look for in results
FRACTIONS = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]

# All models in the study
ALL_MODELS = list(model_families.keys())

# Family colour palette for consistent colouring across plots
FAMILY_COLORS = {'GBDT': '#2196F3', 'DL': '#FF9800', 'FM': '#4CAF50'}
MODEL_FAMILY_MAP = model_families  # already defined above

# Collect learning curve data from results/raw/<dataset>/<model>/learning_curve.json
lc_records = []
raw_root = '../results/raw'

for dataset_path in sorted(glob_module.glob(f'{raw_root}/*')):
    dataset_name = os.path.basename(dataset_path)
    for model_name in ALL_MODELS:
        lc_path = os.path.join(dataset_path, model_name, 'learning_curve.json')
        if not os.path.exists(lc_path):
            continue
        with open(lc_path) as f:
            lc = json.load(f)
        for frac, score in zip(lc['fractions'], lc['scores']):
            lc_records.append({
                'dataset': dataset_name,
                'model': model_name,
                'family': MODEL_FAMILY_MAP.get(model_name, 'Unknown'),
                'fraction': frac,
                'score': score,
            })

if not lc_records:
    print("No learning_curve.json files found under results/raw/.")
    print("This section is a TEMPLATE — run the pipeline with learning-curve mode enabled first.")
    print("\nTo generate learning curves, train each model on subsampled fractions and save:")
    print("  json.dump({'fractions': FRACTIONS, 'scores': scores}, open(lc_path, 'w'))")
else:
    lc_df = pd.DataFrame(lc_records)

    # Plot per-dataset learning curves, one subplot per dataset, lines coloured by family
    datasets_with_lc = lc_df['dataset'].unique()
    n_ds = len(datasets_with_lc)
    ncols = min(3, n_ds)
    nrows = (n_ds + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), squeeze=False)

    for idx, ds in enumerate(sorted(datasets_with_lc)):
        ax = axes[idx // ncols][idx % ncols]
        ds_df = lc_df[lc_df['dataset'] == ds]

        for model_name, grp in ds_df.groupby('model'):
            family = MODEL_FAMILY_MAP.get(model_name, 'Unknown')
            color = FAMILY_COLORS.get(family, 'grey')
            grp_sorted = grp.sort_values('fraction')
            ax.plot(grp_sorted['fraction'], grp_sorted['score'],
                    marker='o', label=f"{model_name} ({family})",
                    color=color, alpha=0.8)

        ax.set_xlabel('Fraction of Training Data')
        ax.set_ylabel('Score')
        ax.set_title(f'Learning Curve — {ds}')
        ax.set_xticks(FRACTIONS)
        ax.set_xlim(0, 1.05)
        ax.legend(fontsize=8, ncol=2)

    # Hide unused subplots
    for idx in range(n_ds, nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    plt.suptitle('True Learning Curves: Performance vs Training Fraction', fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig('../results/figures/learning_curves.png', dpi=300, bbox_inches='tight')
    plt.savefig('../results/figures/learning_curves.pdf', bbox_inches='tight')
    plt.show()

    # Summary: average slope (data efficiency) per model family
    lc_agg = (
        lc_df.groupby(['model', 'family', 'fraction'])['score']
        .mean()
        .reset_index()
    )
    print("\nMean score by family and fraction:")
    family_curve = lc_agg.groupby(['family', 'fraction'])['score'].mean().unstack('fraction')
    print(family_curve.to_string())

## Rank Stability Analysis

How stable are model rankings across the 5 outer CV folds?
A model with low rank variance is consistently ranked the same regardless of which data split is used — a sign of robustness.
High rank variance means the model's relative performance depends heavily on the particular data split.

In [ ]:
# Rank Stability Analysis: per-fold ranks across outer CV folds
# Expects fold_results.csv to have columns: model, dataset, fold, and a performance metric.
# Ranks are computed per (dataset, fold) so that rank 1 = best model on that fold.

try:
    fold_df = pd.read_csv('../results/aggregated/fold_results.csv')

    # Determine the primary metric to rank on.
    # We prefer roc_auc for binary, accuracy for multiclass, r2 for regression.
    # Use whatever metric columns are available.
    candidate_metrics = ['roc_auc', 'accuracy', 'r2', 'rmse']
    rank_metric = next((m for m in candidate_metrics if m in fold_df.columns), None)

    if rank_metric is None:
        print("No recognised metric column found in fold_results.csv. "
              "Expected one of: roc_auc, accuracy, r2, rmse.")
    else:
        # For RMSE lower is better, so negate before ranking
        higher_is_better = rank_metric != 'rmse'
        ascending_rank = not higher_is_better  # rank(ascending=True) → rank 1 = smallest

        # Compute per-(dataset, fold) rank of each model
        fold_df['rank'] = (
            fold_df
            .groupby(['dataset', 'fold'])[rank_metric]
            .rank(ascending=ascending_rank, method='min')
        )

        # Box plot: rank distribution per model (lower rank = better)
        model_order = (
            fold_df.groupby('model')['rank']
            .median()
            .sort_values()
            .index.tolist()
        )

        fig, ax = plt.subplots(figsize=(14, 6))
        fold_df.boxplot(
            column='rank',
            by='model',
            ax=ax,
            order=model_order,
            patch_artist=True,
            boxprops=dict(facecolor='steelblue', alpha=0.6),
            medianprops=dict(color='red', linewidth=2),
            whiskerprops=dict(linestyle='--'),
        )
        ax.set_title(f'Rank Stability Across Outer CV Folds\n(metric: {rank_metric}, rank 1 = best)')
        ax.set_xlabel('Model')
        ax.set_ylabel('Rank (per dataset-fold)')
        plt.suptitle('')  # Remove auto-generated Pandas title
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.savefig('../results/figures/rank_stability.png', dpi=300, bbox_inches='tight')
        plt.savefig('../results/figures/rank_stability.pdf', bbox_inches='tight')
        plt.show()

        print(f"\nMedian rank per model (metric={rank_metric}):")
        print(fold_df.groupby('model')['rank'].agg(['median', 'std', 'min', 'max']).sort_values('median'))

except FileNotFoundError:
    print("fold_results.csv not found at ../results/aggregated/fold_results.csv.")
    print("Run the full pipeline first (scripts/run_all.py), then re-execute this cell.")
except Exception as e:
    print(f"Rank stability analysis failed: {e}")